In [ ]:
#  run once
!nvidia-smi
!pip install -q --upgrade ultralytics timm albumentations==1.3.1 torchmetrics==0.11.4 scikit-learn matplotlib pandas


Sun Sep 21 12:03:07 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install pandas==2.2.2 --quiet


In [ ]:
!pip install -q timm albumentations==1.3.1 torchmetrics==0.11.4

In [ ]:
!pip install ultralytics --quiet



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 32.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.flush_and_unmount()


Drive not mounted, so nothing to flush and unmount.


In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

# ==============================
# 1. Mount Google Drive
# ==============================
from google.colab import drive
drive.mount('/content/drive')

# ==============================
# 2. Dataset Path (Shortcut in MyDrive)
# ==============================
base_dir = "/content/drive/MyDrive/Smart_Food_Analyzer-Food_Images/Primary Categories"

if not os.path.exists(base_dir):
    raise FileNotFoundError(f"❌ Dataset not found at {base_dir}. Please check your shortcut path.")

print("✅ Found dataset at:", base_dir)

# ==============================
# 3. Prepare Dataset for YOLOv8
# ==============================
dataset_dir = "/content/dataset_cls"
train_dir = os.path.join(dataset_dir, "train")
val_dir = os.path.join(dataset_dir, "val")

os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

# Loop through primary categories (Rice, Meat, etc.)
for primary_cat in os.listdir(base_dir):
    primary_path = os.path.join(base_dir, primary_cat)
    if not os.path.isdir(primary_path):
        continue

    # Loop through food-item subfolders (Biryani, Butter Chicken, etc.)
    for food_item in os.listdir(primary_path):
        food_item_path = os.path.join(primary_path, food_item)
        if not os.path.isdir(food_item_path):
            continue

        # Collect only valid images
        images = [f for f in os.listdir(food_item_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if len(images) == 0:
            continue  # skip empty folders

        # Split train/val (80/20)
        train_imgs, val_imgs = train_test_split(images, test_size=0.2, random_state=42)

        # Make class subfolders
        os.makedirs(os.path.join(train_dir, food_item), exist_ok=True)
        os.makedirs(os.path.join(val_dir, food_item), exist_ok=True)

        # Copy files
        for img in train_imgs:
            shutil.copy(os.path.join(food_item_path, img), os.path.join(train_dir, food_item, img))
        for img in val_imgs:
            shutil.copy(os.path.join(food_item_path, img), os.path.join(val_dir, food_item, img))


Mounted at /content/drive
✅ Found dataset at: /content/drive/MyDrive/Smart_Food_Analyzer-Food_Images/Primary Categories


In [ ]:
# Download YOLOv9 classification checkpoint
!wget https://github.com/WongKinYiu/yolov9/releases/download/v0.1/yolov9-c.pt



--2025-09-21 13:35:14--  https://github.com/WongKinYiu/yolov9/releases/download/v0.1/yolov9-c.pt
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/759338070/c8ca43f2-0d2d-4aa3-a074-426505bfbfb1?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-09-21T14%3A08%3A31Z&rscd=attachment%3B+filename%3Dyolov9-c.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-09-21T13%3A07%3A52Z&ske=2025-09-21T14%3A08%3A31Z&sks=b&skv=2018-11-09&sig=BDukwm%2FKNePRKaDPl1JlKZ1MJzxB5m2WfIKw%2FOBcMYU%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc1ODQ2MjAxNCwibmJmIjoxNzU4NDYxNzE0LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5ibG9iLm

In [ ]:
!git clone https://github.com/WongKinYiu/yolov9.git
%cd yolov9
!pip install -r requirements.txt


Cloning into 'yolov9'...
remote: Enumerating objects: 781, done.
remote: Counting objects: 100% (316/316), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 781 (delta 265), reused 257 (delta 257), pack-reused 465 (from 1)
Receiving objects: 100% (781/781), 3.25 MiB | 6.21 MiB/s, done.
Resolving deltas: 100% (339/339), done.
/content/yolov9
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 79.6 MB/s eta 0:00:00


In [ ]:
%%bash
cd /content/yolov9

if [ ! -f yolov9-c.pt ]; then
  echo "Downloading yolov9-c.pt..."
  wget -q --show-progress https://github.com/WongKinYiu/yolov9/releases/download/v0.1/yolov9-c.pt -O yolov9-c.pt
else
  echo "yolov9-c.pt already exists"
fi

ls -lh yolov9-c.pt


-rw-r--r-- 1 root root 99M Feb 18  2024 yolov9-c.pt



     0K .......... .......... .......... .......... ..........  0% 31.4M 3s
    50K .......... .......... .......... .......... ..........  0%  169M 2s
   100K .......... .......... .......... .......... ..........  0% 64.1M 2s
   150K .......... .......... .......... .......... ..........  0%  240M 1s
   200K .......... .......... .......... .......... ..........  0% 85.4M 1s
   250K .......... .......... .......... .......... ..........  0%  185M 1s
   300K .......... .......... .......... .......... ..........  0%  228M 1s
   350K .......... .......... .......... .......... ..........  0%  200M 1s
   400K .......... .......... .......... .......... ..........  0%  299M 1s
   450K .......... .......... .......... .......... ..........  0%  172M 1s
   500K .......... .......... .......... .......... ..........  0%  254M 1s
   550K .......... .......... .......... .......... ..........  0%  213M 1s
   600K .......... .......... .......... .......... ..........  0%  246M 1s
   650K ...

In [ ]:
!nvidia-smi
!python -c "import torch; print('torch.cuda.is_available()', torch.cuda.is_available()); print('torch.__version__', torch.__version__)"


Sun Sep 21 14:01:15 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             11W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%%bash
cd /content/yolov9

DATA_DIR="/content/dataset_cls"  # must contain train/ and val/
EPOCHS=20
BATCH=32
IMG=224
DEVICE=0

# Train from scratch (no pretrained weights)
WANDB_MODE=disabled python classify/train.py \
    --model torch_load \
    --data $DATA_DIR \
    --epochs $EPOCHS \
    --batch-size $BATCH \
    --img $IMG \
    --device $DEVICE


E0000 00:00:1758463984.852449   40001 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758463984.858555   40001 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1758463984.873900   40001 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1758463984.873925   40001 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1758463984.873928   40001 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1758463984.873930   40001 computation_placer.cc:177] computation placer already registered. Please check linka

CalledProcessError: Command 'b'cd /content/yolov9\n\nDATA_DIR="/content/dataset_cls"  # must contain train/ and val/\nEPOCHS=20\nBATCH=32\nIMG=224\nDEVICE=0\n\n# Train from scratch (no pretrained weights)\nWANDB_MODE=disabled python classify/train.py \\\n    --model torch_load \\\n    --data $DATA_DIR \\\n    --epochs $EPOCHS \\\n    --batch-size $BATCH \\\n    --img $IMG \\\n    --device $DEVICE\n'' returned non-zero exit status 1.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os

# ==============================
# 1. Check device
# ==============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ==============================
# 2. Paths
# ==============================
train_dir = "/content/dataset_cls/train"
val_dir = "/content/dataset_cls/val"

# ==============================
# 3. Data Transforms
# ==============================
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ==============================
# 4. Datasets & Loaders
# ==============================
train_dataset = datasets.ImageFolder(train_dir, transform=transform_train)
val_dataset = datasets.ImageFolder(val_dir, transform=transform_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

num_classes = len(train_dataset.classes)
print("Number of classes:", num_classes)

# ==============================
# 5. Load MobileNetV3 Pretrained
# ==============================
model = models.mobilenet_v3_large(pretrained=True)

# Replace classifier
in_features = model.classifier[3].in_features  # final Linear layer input features
model.classifier[3] = nn.Linear(in_features, num_classes)

model = model.to(device)

# ==============================
# 6. Loss & Optimizer
# ==============================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# ==============================
# 7. Training Loop
# ==============================
num_epochs = 30

for epoch in range(num_epochs):
    # Training
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / total
    train_acc = correct / total

    # Validation
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss /= total
    val_acc = correct / total

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

# ==============================
# 8. Save Model
# ==============================
torch.save(model.state_dict(), "/content/drive/MyDrive/mobilenetv3_food.pth")
print("✅ Model saved as mobilenetv3_food.pth")



Using device: cpu
Number of classes: 27


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Large_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-8738ca79.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 48.6MB/s]


Epoch [1/30] Train Loss: 2.0457 | Train Acc: 0.4610 | Val Loss: 1.2122 | Val Acc: 0.6510
Epoch [2/30] Train Loss: 0.8295 | Train Acc: 0.7614 | Val Loss: 0.7056 | Val Acc: 0.7894
Epoch [3/30] Train Loss: 0.5246 | Train Acc: 0.8416 | Val Loss: 0.5816 | Val Acc: 0.8303
Epoch [4/30] Train Loss: 0.3597 | Train Acc: 0.8952 | Val Loss: 0.5329 | Val Acc: 0.8520
Epoch [5/30] Train Loss: 0.2605 | Train Acc: 0.9247 | Val Loss: 0.5021 | Val Acc: 0.8520
Epoch [6/30] Train Loss: 0.1836 | Train Acc: 0.9503 | Val Loss: 0.4841 | Val Acc: 0.8652
Epoch [7/30] Train Loss: 0.1269 | Train Acc: 0.9686 | Val Loss: 0.4787 | Val Acc: 0.8628
Epoch [8/30] Train Loss: 0.0934 | Train Acc: 0.9771 | Val Loss: 0.4891 | Val Acc: 0.8652
Epoch [9/30] Train Loss: 0.0673 | Train Acc: 0.9835 | Val Loss: 0.4696 | Val Acc: 0.8700
Epoch [10/30] Train Loss: 0.0461 | Train Acc: 0.9924 | Val Loss: 0.5047 | Val Acc: 0.8652
Epoch [11/30] Train Loss: 0.0522 | Train Acc: 0.9872 | Val Loss: 0.4962 | Val Acc: 0.8652
Epoch [12/30] Train

In [ ]:
import torch
from torchvision import transforms, models
from PIL import Image

# ==============================
# 1. Load the saved model
# ==============================
#num_classes = 10  # replace with your number of classes
#class_names = sorted(os.listdir("/content/dataset_cls/train"))  # auto class names from train folder
class_names = sorted([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])
num_classes = len(class_names)
model = models.mobilenet_v3_large(pretrained=False)
in_features = model.classifier[3].in_features
model.classifier[3] = torch.nn.Linear(in_features, num_classes)

model.load_state_dict(torch.load("/content/drive/MyDrive/mobilenetv3_food.pth", map_location="cpu"))
model.eval()

# ==============================
# 2. Image transforms
# ==============================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

# ==============================
# 3. Load & preprocess image
# ==============================
image_path = "/content/drive/MyDrive/image_4.jpg"  # your test image path
image = Image.open(image_path).convert("RGB")
image_tensor = transform(image).unsqueeze(0)  # add batch dimension

# ==============================
# 4. Make prediction
# ==============================
with torch.no_grad():
    outputs = model(image_tensor)
    _, predicted = torch.max(outputs, 1)
    class_name = class_names[predicted.item()]

print(f"✅ Predicted class: {class_name}")


✅ Predicted class: Chicken_Shorba
